"""
# Bonsai 2 on TPU silicon (JAX forward pass)

Notebooks 01-03 built the two primitives this one needs: a working GGUF
`prism.hadamard.*` reader (02) and a verified signed-FWHT (03). This is
the next rung: a complete JAX forward pass for the Bonsai 2 27B (Qwen3.5
hybrid dense-attention + gated-delta-net, PTQ1_0 ternary weights),
running **on the TPU**, greedy-decoding a prompt.

llama.cpp is deliberately not used: it has no TPU backend, so on a TPU VM
it would silently run the graph on the host CPU. Every op below is a
`jnp` op on a `jax.devices()` TPU.

Every rule is ported from `PrismML-Eng/llama.cpp` branch `prism`, not
guessed -- `file:line` is given in each comment.

| Rule | Source |
| --- | --- |
| block: `qs[24]`, `qh[2]`, fp16 `d`, 28 B per 128 weights | `ggml-common.h:214` |
| trit = `((q*3^n)*3)>>8 - 1`, trit-plane value order | `arch/arm/quants.c:1532` |
| fold `x' = R @ (x*sign)`; latent embedding `h = (R@z)*sign` | `llama-graph.cpp:1596,2453` |
| `ssm_out` pre-permute `[hd,nk,rep] -> [hd,rep,nk]` | `llama-model.cpp:2135` |
| GDN step: decay, read, delta, outer product, `/sqrt(128)` | `ggml-cpu/ops.cpp:11060` |
| causal conv `y = sum_k w[k]*x[t-3+k]`, then SiLU | `ggml-cpu/ops.cpp:9776` |
| RMS norm = plain multiply by `w` (no `1+w`) | `llama-graph.cpp:1729` |
| Q and gate interleaved in `attn_q` | `models/qwen35.cpp:357` |
| gated norm = `silu(z) * rms_norm(o, ssm_norm)` | `models/qwen35.cpp:324` |

**Verified against what:** greedy tokens must answer the prompt in fluent
English. A wrong transpose, sign order, rope convention or GDN indexing
produces noise, so fluency is a real check, not a formality.
"""


In [ ]:
import os
import time

import jax
import jax.numpy as jnp
import numpy as np

# TPU matmuls default to reduced (bf16) accumulation even for f32 inputs.
# The ternary weights are exactly +-1, but the folded projections and the
# Hadamard transform need full accumulation to match the reference, so force
# the highest available precision for every dot in this file.
jax.config.update("jax_default_matmul_precision", "highest")

print(f"jax {jax.__version__} | backend {jax.default_backend()}")
for d in jax.devices():
    print("device:", d.platform, d.id)

GGUF = os.environ.get("GGUF_PATH", "/content/gguf/Ternary-Bonsai-2-27B-PTQ1_0.gguf")
assert os.path.exists(GGUF), f"missing {GGUF}"
N_GEN = int(os.environ.get("N_TOKENS", "64"))
PROMPT = os.environ.get("PROMPT", "Hello")
print(f"file {GGUF} ({os.path.getsize(GGUF)/2**30:.2f} GiB) | "
      f"prompt {PROMPT!r} | {N_GEN} tokens")


"""
## PTQ1_0 decode

`block_ptq1_0` is 28 bytes: `qs` (24 B, five base-3 digits per byte, 120
values), `qh` (2 B, four digits per byte, 8 values), an fp16 `d`. The
value order inside a block is **trit-plane**, not byte order: `qs[0..15]`
yields values 0-15, 16-31, ... 64-79 for trit index 0-4. A wrong order
still decodes to {-1,0,1}, so the self-test checks hand-computed bytes.
"""


In [ ]:
import struct

QK = 128     # weights per block
BLK = 28     # bytes per block
POW3 = (1, 3, 9, 27, 81)


def _trit_host(q, p):
    v = (q.astype(np.uint32) * np.uint32(p)) % np.uint32(256)
    return (((v + (v >> 1)) >> 1) >> 6).astype(np.int8) - 1


def decode_ptq1_host(raw):
    """[nblocks, 28] uint8 -> [nblocks, 128] float32 (trit * fp16 scale)."""
    raw = raw.reshape(-1, BLK)
    qs, qh = raw[:, :24], raw[:, 24:26]
    d = raw[:, 26:28].copy().view(np.float16).astype(np.float32)[:, 0]
    out = np.zeros((raw.shape[0], QK), dtype=np.float32)
    for i, p in enumerate(POW3):
        out[:, i * 16:(i + 1) * 16] = _trit_host(qs[:, :16], p)
    for i, p in enumerate(POW3):
        out[:, 80 + i * 8:80 + (i + 1) * 8] = _trit_host(qs[:, 16:24], p)
    for i, p in enumerate((1, 3, 9, 27)):
        out[:, 120 + 2 * i] = _trit_host(qh[:, 0], p)
        out[:, 121 + 2 * i] = _trit_host(qh[:, 1], p)
    return out * d[:, None]


def _trit_dev(q, p):
    v = (q * np.int32(p)) & 0xFF
    return ((((v + (v >> 1)) >> 1) >> 6).astype(jnp.float32) - 1.0)


@jax.jit
def decode_ptq1_dev(raw):
    """Same decode as `decode_ptq1_host`, on device."""
    b = raw.reshape(-1, BLK).astype(jnp.int32)
    qs, qh = b[:, :24], b[:, 24:26]
    # fp16 scale, reinterpreted as float16 then widened (NOT <<16: the fp16
    # exponent is 5 bits, so a raw shift silently rescales every weight).
    u16 = (b[:, 26] | (b[:, 27] << 8)).astype(jnp.uint16)
    scale = u16.view(jnp.float16).astype(jnp.float32)
    cols = []
    for p in POW3:
        cols.append(_trit_dev(qs[:, :16], p))
    for p in POW3:
        cols.append(_trit_dev(qs[:, 16:24], p))
    for p in (1, 3, 9, 27):
        cols.append(jnp.stack([_trit_dev(qh[:, 0], p), _trit_dev(qh[:, 1], p)], 1))
    return jnp.concatenate(cols, 1) * scale[:, None]


def _blk(scale=2.0):
    b = np.zeros((1, BLK), dtype=np.uint8)
    sb = np.float16(scale).tobytes()
    b[0, 26], b[0, 27] = sb[0], sb[1]
    return b


# Hand-computed, byte by byte (values are trit * fp16 scale, scale = 2.0).
# Value layout inside a block: index 16*p + b is `qs[b]` at multiplier 3^p,
# for p in 0..4 (so index 0-15 = bytes 0-15 at multiplier 1). qh[0] lands
# at indices 120,122,124,126 for multipliers 1,3,9,27.
assert np.all(decode_ptq1_host(_blk()) == -2.0)          # zero -> every trit -1
_b = _blk(); _b[0, 0] = 0xD5                              # 213 -> trit(1)=+1
assert decode_ptq1_host(_b)[0, 0] == 2.0
assert decode_ptq1_host(_b)[0, 16] == 0.0                # same byte, trit(3)=0
assert decode_ptq1_host(_b)[0, 1] == -2.0                # byte 1 is still zero
_b = _blk(); _b[0, 0] = 0x2A                             # 42 -> trit(1)=-1
assert decode_ptq1_host(_b)[0, 0] == -2.0
assert decode_ptq1_host(_b)[0, 64] == -2.0               # trit(81)=-1
assert decode_ptq1_host(_b)[0, 48] == 0.0                # trit(27)=0
_b = _blk(); _b[0, 24] = 0x2A                            # qh[0], trit(1)=-1
assert decode_ptq1_host(_b)[0, 120] == -2.0
_b = _blk(); _b[0, 24] = 0xFF                            # qh[0], trit(27)=+1
assert decode_ptq1_host(_b)[0, 126] == 2.0
print("host decode self-test PASS (all 5 trit positions, qh, zero, saturation)")

_r = np.zeros((5, BLK), np.uint8)
for i in range(5):
    _r[i, 0] = (0xD5, 0x2A, 0x00, 0xFF, 0x55)[i]
    _r[i, 24] = (0x2A, 0x55, 0xD5, 0x01, 0x03)[i]
    _sb = np.float16(0.5 * i).tobytes()
    _r[i, 26] = _sb[0]
    _r[i, 27] = _sb[1]
assert np.array_equal(np.asarray(decode_ptq1_dev(jnp.asarray(_r))),
                      decode_ptq1_host(_r)), "device decode != host decode"
print("device decode matches host decode exactly")


"""
## Weight store

Ternary bytes stay on the TPU and are decoded **one layer at a time**: a
fully decoded 27 B ternary model is ~54 GB in f32 against ~16 GiB of
v5e-1 HBM. Small tensors (norms, gates, conv kernels) are cached whole.
"""


In [ ]:
DTYPE = {0: ("F32", 4, 1), 1: ("F16", 2, 1), 30: ("BF16", 2, 1),
         143: ("PTQ1_0", BLK, QK)}


def _rd(fh, n):
    b = fh.read(n)
    assert len(b) == n, "short read"
    return b


def _s(fh):
    (n,) = struct.unpack("<Q", _rd(fh, 8))
    return _rd(fh, n).decode("utf-8", "replace")


class GGUFReader:
    def __init__(self, path):
        self.fh = open(path, "rb")
        assert _rd(self.fh, 4) == b"GGUF"
        (self.version,) = struct.unpack("<I", _rd(self.fh, 4))
        (self.n_tensors,) = struct.unpack("<Q", _rd(self.fh, 8))
        (self.n_kv,) = struct.unpack("<Q", _rd(self.fh, 8))
        self.meta = {}
        for _ in range(self.n_kv):
            k = _s(self.fh)
            (vt,) = struct.unpack("<I", _rd(self.fh, 4))
            self.meta[k] = self._val(vt)
        self.info = {}
        for _ in range(self.n_tensors):
            nm = _s(self.fh)
            (nd,) = struct.unpack("<I", _rd(self.fh, 4))
            dims = list(struct.unpack(f"<{nd}Q", _rd(self.fh, 8 * nd)))
            (dt,) = struct.unpack("<I", _rd(self.fh, 4))
            (off,) = struct.unpack("<Q", _rd(self.fh, 8))
            self.info[nm] = (dims, dt, off)
        end = self.fh.tell()
        self.filesize = os.fstat(self.fh.fileno()).st_size
        self.data_start = None
        for al in (32, 64, 16, 8, 1):
            cand = (end + al - 1) // al * al
            nm = max(self.info, key=lambda k: self.info[k][2])
            if cand + self.info[nm][2] + self._size(self.info[nm]) == self.filesize:
                self.data_start = cand
                break
        assert self.data_start, "tensor data section not found"

    def _val(self, vt):
        fh = self.fh
        if vt == 8:
            return _s(fh)
        if vt == 9:
            (et,) = struct.unpack("<I", _rd(fh, 4))
            (n,) = struct.unpack("<Q", _rd(fh, 8))
            if et == 8:
                return [_s(fh) for _ in range(n)]
            fmt = {0: "B", 1: "b", 2: "H", 3: "h", 4: "I", 5: "i",
                   6: "f", 7: "?", 10: "Q", 11: "q", 12: "d"}[et]
            return list(struct.unpack(f"<{n}{fmt}", _rd(fh, n * struct.calcsize(fmt))))
        fmt = {0: "B", 1: "b", 2: "H", 3: "h", 4: "I", 5: "i",
               6: "f", 7: "?", 10: "Q", 11: "q", 12: "d"}[vt]
        return struct.unpack("<" + fmt, _rd(fh, struct.calcsize(fmt)))[0]

    def _size(self, t):
        dims, dt, _ = t
        _, bb, be = DTYPE[dt]
        return (int(np.prod(dims)) + be - 1) // be * bb

    def read(self, name):
        dims, dt, off = self.info[name]
        self.fh.seek(self.data_start + off)
        return _rd(self.fh, self._size((dims, dt, off))), dims, dt


ws = GGUFReader(GGUF)
print(f"GGUF v{ws.version}: {ws.n_tensors} tensors, {len(ws.meta)} keys, "
      f"data@{ws.data_start}")

TERN = ("attn_qkv.weight", "attn_gate.weight", "attn_q.weight",
        "attn_k.weight", "attn_v.weight", "attn_output.weight",
        "ffn_gate.weight", "ffn_up.weight", "ffn_down.weight", "ssm_out.weight")
SMALL = ("attn_norm.weight", "post_attention_norm.weight",
         "attn_q_norm.weight", "attn_k_norm.weight", "ssm_norm.weight",
         "ssm_conv1d.weight", "ssm_dt.bias", "ssm_a", "ssm_beta.weight",
         "ssm_alpha.weight")

def _load_dense(raw, dims, dt):
    """Decode one non-ternary tensor. GGUF dims[0] is the CONTIGUOUS dim, so
    element (i0, i1, ...) is at flat[i0 + i1*dims[0] + ...] -- Fortran order.
    Reshaping in C order silently transposes every 2-D weight while leaving
    1-D tensors (norms, biases) correct, which is exactly the failure mode
    this notebook debugged against llama.cpp."""
    n = int(np.prod(dims))
    if dt == 143:
        arr = decode_ptq1_host(np.frombuffer(raw, np.uint8)).reshape(-1)
    elif dt == 30:
        arr = ((np.frombuffer(raw, np.uint16).astype(np.uint32) << 16)
               .view(np.float32).reshape(-1))
    elif dt == 0:
        arr = np.frombuffer(raw, np.float32).reshape(-1)
    elif dt == 1:
        arr = np.frombuffer(raw, np.float16).astype(np.float32).reshape(-1)
    else:
        raise ValueError(f"unhandled dtype {dt}")
    return jnp.asarray(np.asarray(arr[:n], np.float32)
                       .reshape(dims, order="F"))


def decode_layer(il):
    """Layer il's ternary projections, Fortran-reshaped to [in, out]."""
    flat = decode_ptq1_dev(LAYER_RAW[il]).reshape(-1)
    return {k: flat[o:o + n].reshape(dims, order="F")
            for k, (o, n, dims) in LAYER_MAP[il].items()}


n_layers = ws.meta["qwen35.block_count"]
LAYER_RAW, LAYER_MAP = [], []
SMALLW = [{} for _ in range(n_layers)]
for il in range(n_layers):
    pre, chunks, cmap, off = f"blk.{il}.", [], {}, 0
    for short in TERN:
        nm = pre + short
        if nm not in ws.info:
            continue
        raw, dims, dt = ws.read(nm)
        assert dt == 143, f"{nm} is {DTYPE[dt][0]}, expected PTQ1_0"
        chunks.append(np.frombuffer(raw, np.uint8).copy())
        cmap[short] = (off, len(raw) // BLK * QK, dims)
        off += len(raw) // BLK * QK
    blob = np.concatenate(chunks) if chunks else np.zeros(0, np.uint8)
    del chunks
    LAYER_RAW.append(jnp.asarray(blob))
    LAYER_MAP.append(cmap)
    del blob
    for short in SMALL:
        nm = pre + short
        if nm not in ws.info:
            continue
        raw, dims, dt = ws.read(nm)
        n = int(np.prod(dims))
        if dt == 143:
            arr = decode_ptq1_host(np.frombuffer(raw, np.uint8)).reshape(-1)
        elif dt == 30:
            arr = ((np.frombuffer(raw, np.uint16).astype(np.uint32) << 16)
                   .view(np.float32).reshape(-1))
        elif dt == 0:
            arr = np.frombuffer(raw, np.float32).reshape(-1)[:n]
        else:
            arr = np.frombuffer(raw, np.float16).astype(np.float32).reshape(-1)[:n]
        SMALLW[il][short] = _load_dense(raw, dims, dt)

out_raw, out_dims, _ = ws.read("output.weight")
emb_raw, emb_dims, _ = ws.read("token_embd.weight")
EMB_BLK = emb_dims[0] // QK
OUT_RAW = jnp.asarray(np.frombuffer(out_raw, np.uint8).copy())
EMB_RAW = jnp.asarray(np.frombuffer(emb_raw, np.uint8).copy())
del out_raw, emb_raw
print(f"output.weight {out_dims}, token_embd {emb_dims} ({EMB_BLK} blocks/row)")
print(f"staged {n_layers} layers: {sum(len(c) for c in LAYER_MAP)} ternary "
      f"+ {len(SMALLW)} cached tensors; "
      f"total TPU bytes {OUT_RAW.nbytes + EMB_RAW.nbytes + sum(r.nbytes for r in LAYER_RAW):,}")


"""
## Fold, sign vectors, rotation

`R = H_1024/sqrt(1024)`; folded matmul `y = W @ (R @ (x*sign))`; the
latent embedding un-folds with the *same* matrix because `R@R = I`.
"""


In [ ]:
BLOCK = 1024


def rotation_matrix(block=BLOCK):
    idx = np.arange(block, dtype=np.uint32)
    parity = np.zeros((block, block), dtype=np.uint8)
    for r in range(block):
        p = np.uint32(r) & idx
        for sh in (16, 8, 4, 2, 1):
            p = p ^ (p >> np.uint32(sh))
        parity[r] = (p & np.uint32(1)).astype(np.uint8)
    return np.where(parity == 1, -1.0, 1.0) / np.sqrt(block)


R = jnp.asarray(rotation_matrix())
assert float(jnp.abs(R @ R - jnp.eye(BLOCK)).max()) < 1e-5

SIGNS, _o = {}, 0
for _w in ws.meta["prism.hadamard.sign_widths"]:
    w = int(_w)
    SIGNS[w] = jnp.asarray(np.asarray(
        ws.meta["prism.hadamard.sign_values"][_o:_o + w], np.float32))
    _o += w
assert _o == len(ws.meta["prism.hadamard.sign_values"])


@jax.jit
def _blocks(x):
    """[n_in] -> [n_blocks, 1024]."""
    return x.reshape(-1, BLOCK)


@jax.jit
def fold(x, sign):
    """[n_in] -> R @ (x * sign), applied per 1024-block."""
    return ((_blocks(x * sign)) @ R.T).reshape(-1)


@jax.jit
def unfold(z, sign):
    """latent row [n_in] -> h = (R @ z) * sign."""
    return (_blocks(z) @ R.T).reshape(-1) * sign


_z = jax.random.normal(jax.random.key(7), (5120,), dtype=jnp.float32)
_sn = jax.random.choice(jax.random.key(9), jnp.array([-1.0, 1.0], jnp.float32),
                        (5120,))
# unfold(fold(x)) must be x: the fold applies (x*s) through R, the inverse
# applies R again (R@R = I) and multiplies by s once more.
_rt = jnp.abs(unfold(fold(_z, _sn), _sn) - _z).max()
_scale = jnp.abs(_z).max()
print(f"unfold(fold(z)) round-trip err {float(_rt):.2e} "
      f"(max|z| = {float(_scale):.2f})")
# TPU matmul precision is reduced vs CPU even at 'highest', so the round-trip
# tolerance is relative and looser than the CPU-only 1e-3 check.
assert float(_rt) < 5e-3 * float(_scale), f"round-trip error too large: {_rt}"
print(f"R orthonormal, max|R@R - I| = "
      f"{float(jnp.abs(R @ R - jnp.eye(BLOCK)).max()):.1e}")
print("rope sections:", ws.meta.get("qwen35.rope.dimension_sections"),
      "| rope type: QWEN35 -> IMROPE (half-split pairs). For text-only input "
      "all positions are equal, so MRoPE reduces to plain RoPE over n_rot "
      "dims with theta_k = base^(-2k/n_rot).")
print(f"sign vectors for widths {sorted(SIGNS)}")


"""
## Configuration, read from the file
"""


In [ ]:
m = ws.meta
cfg = {
    "n_layer": m["qwen35.block_count"],
    "n_embd": m["qwen35.embedding_length"],
    "n_ff": m["qwen35.feed_forward_length"],
    "n_head": m["qwen35.attention.head_count"],
    "n_head_kv": m["qwen35.attention.head_count_kv"],
    "head_dim": m["qwen35.attention.key_length"],
    "n_rot": int(m.get("qwen35.rope.dimension_count", 0)
                  or m["qwen35.attention.key_length"]),
    "freq_base": m["qwen35.rope.freq_base"],
    "eps": m["qwen35.attention.layer_norm_rms_epsilon"],
    "attn_interval": m["qwen35.full_attention_interval"],
    "conv": m["qwen35.ssm.conv_kernel"],
    "state": m["qwen35.ssm.state_size"],
    "dt_rank": m["qwen35.ssm.time_step_rank"],
    "n_group": m["qwen35.ssm.group_count"],
    "vocab": int(emb_dims[-1]),
    "rope_sections": list(m.get("qwen35.rope.dimension_sections", [])),
}
cfg["n_k"] = cfg["n_group"]
cfg["n_v"] = cfg["dt_rank"]
cfg["rep"] = cfg["n_v"] // cfg["n_k"]
cfg["key_dim"] = cfg["state"] * cfg["n_k"]
cfg["val_dim"] = cfg["state"] * cfg["n_v"]
cfg["conv_dim"] = 2 * cfg["key_dim"] + cfg["val_dim"]
cfg["is_recr"] = [((i + 1) % cfg["attn_interval"] != 0) and i < cfg["n_layer"]
                  for i in range(cfg["n_layer"])]
print(f"{cfg['n_layer']} layers: {sum(cfg['is_recr'])} gated-delta-net + "
      f"{cfg['n_layer'] - sum(cfg['is_recr'])} full-attention")
print(f"n_embd {cfg['n_embd']}  n_ff {cfg['n_ff']}  vocab {cfg['vocab']}")
print(f"gdn: {cfg['n_k']}k x {cfg['state']} / {cfg['n_v']}v x {cfg['state']} "
      f"(rep {cfg['rep']}), conv {cfg['conv']} x {cfg['conv_dim']}")
print(f"attn: {cfg['n_head']}q / {cfg['n_head_kv']}kv x {cfg['head_dim']}, "
      f"rope {cfg['n_rot']} dims @ {cfg['freq_base']:g}")


"""
## Tokenizer (GPT-2 BPE, from the file's own tables)
"""


In [ ]:
import re


def _bytes_to_unicode():
    bs = (list(range(ord("!"), ord("~") + 1)) + list(range(0xA1, 0xAD))
          + list(range(0xAE, 0x100)))
    cs, n = bs[:], 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, (chr(c) for c in cs)))


class Tokenizer:
    PAT = re.compile(r"'s|'t|'re|'ve|'m|'ll|'d| ?[^\W\d_]+| ?\d+| "
                     r"?[^\s\w]+|\s+(?!\S)|\s+")

    def __init__(self, meta):
        toks = meta["tokenizer.ggml.tokens"]
        ttype = np.asarray(meta["tokenizer.ggml.token_type"])
        self.tok2id = {t: i for i, t in enumerate(toks)}
        self.id2tok = toks
        self.rank = {}
        for r, mm in enumerate(meta["tokenizer.ggml.merges"]):
            a, _, b = mm.partition(" ")
            self.rank[(a, b)] = r
        self.enc = _bytes_to_unicode()
        self.dec = {v: k for k, v in self.enc.items()}
        self.byte_id = {}
        for i in np.where(ttype == 3)[0]:
            t = toks[i]
            if len(t) == 6 and t.startswith("<0x") and t.endswith(">"):
                self.byte_id[int(t[3:5], 16)] = int(i)
        self.bos = int(meta.get("tokenizer.ggml.bos_token_id", 248044))
        self.eos = int(meta.get("tokenizer.ggml.eos_token_id", 248046))
        self.add_bos = int(meta.get("tokenizer.ggml.add_bos_token", 0)) == 1

    def bpe(self, token):
        word = list(token)
        while len(word) > 1:
            best, bi = None, -1
            for i in range(len(word) - 1):
                r = self.rank.get((word[i], word[i + 1]))
                if r is not None and (best is None or r < best):
                    best, bi = r, i
            if bi < 0:
                break
            word = word[:bi] + [word[bi] + word[bi + 1]] + word[bi + 2:]
        return word

    def encode(self, text):
        ids = [self.bos] if self.add_bos else []
        for piece in self.PAT.findall(text):
            mapped = "".join(self.enc[b] for b in piece.encode("utf-8"))
            for t in self.bpe(mapped):
                ids.append(self.tok2id[t] if t in self.tok2id
                           else self.byte_id[t.encode("utf-8")[0]])
        return ids

    def decode(self, ids):
        out = bytearray()
        for i in ids:
            t = self.id2tok[int(i)]
            if len(t) == 6 and t.startswith("<0x") and t.endswith(">"):
                out.append(int(t[3:5], 16))
            else:
                out.extend(self.dec[c] for c in t if c in self.dec)
                out.extend(ord(c) for c in t if c not in self.dec)
        return bytes(out).decode("utf-8", "replace")


tok = Tokenizer(ws.meta)
_ids = tok.encode("Hello")
assert tok.decode(_ids) == "Hello", tok.decode(_ids)
print(f"encode('Hello') -> {_ids}; ids -> {[tok.id2tok[i] for i in _ids]}")
print("tokenizer round-trip PASS")


"""
## The forward pass

Residual structure shared by both layer kinds. Folded projections are
`x_fold @ W` with `W` held as `[n_in, n_out]` (GGUF `ne[0]` is the input),
so `y = x @ W` needs no transpose. States: one `[state, state, n_v]`
delta-net matrix per GDN layer, and a KV cache per full-attention layer.
"""


In [ ]:
@jax.jit
def rms(x, w, eps):
    return x * jax.lax.rsqrt(jnp.mean(x * x, -1, keepdims=True) + eps) * w


def rope_neox(x, pos, n_rot, base):
    """x: [heads, dim]. Half-split pairs (j, j + n_rot/2) -- the QWEN35
    IMROPE convention (ggml-cpu/ops.cpp:6091 + 5944)."""
    half = n_rot // 2
    inv = base ** (-jnp.arange(half, dtype=jnp.float32) * 2.0 / n_rot)
    ang = jnp.asarray(pos, jnp.float32) * inv
    c, s = jnp.cos(ang), jnp.sin(ang)
    lo, hi = x[..., :half], x[..., half:2 * half]
    y = jnp.concatenate([lo * c - hi * s, lo * s + hi * c], -1)
    return y if 2 * half == x.shape[-1] else jnp.concatenate([y, x[..., 2 * half:]], -1)


def rope_plain(x, pos, n_rot, base):
    """x: [heads, dim]. Adjacent pairs (2i, 2i+1) -- kept as an A/B switch."""
    half = n_rot // 2
    inv = base ** (-jnp.arange(half, dtype=jnp.float32) * 2.0 / n_rot)
    ang = jnp.asarray(pos, jnp.float32) * inv
    c, s = jnp.cos(ang), jnp.sin(ang)
    xr = x[..., :2 * half].reshape(x.shape[:-1] + (half, 2))
    x0, x1 = xr[..., 0], xr[..., 1]
    y = jnp.stack([x0 * c - x1 * s, x0 * s + x1 * c], -1
                  ).reshape(x.shape[:-1] + (2 * half,))
    return y if 2 * half == x.shape[-1] else jnp.concatenate([y, x[..., 2 * half:]], -1)


@jax.jit
def l2n(x, eps):
    """ggml l2_norm: scale = 1/max(sqrt(sum_sq), eps), taken over axis 0.
    For the GDN qk view [head_dim=128, 2*n_k_heads=32], that means EACH HEAD
    (a 128-length column) is normalised independently. Normalising the wrong
    axis silently wrecks every downstream product. (ops.cpp:4209)"""
    r = jnp.sqrt(jnp.sum(x * x, axis=0, keepdims=True))
    return x / jnp.maximum(r, eps)


def make_rope(kind="neox"):
    """Bake the model's n_rot / freq_base into a jitted rope of (x, pos)."""
    n_rot, base = cfg["n_rot"], cfg["freq_base"]
    fn = rope_plain if kind == "plain" else rope_neox

    @jax.jit
    def _rope(x, pos):
        return fn(x, pos, n_rot, base)

    return _rope


def ggml_view(flat, d0, d1):
    """View a flat [..., d0*d1] array as ggml [d0, d1]: element (i0, i1) is
    flat[i0 + i1*d0]. In numpy/JAX C-order that is reshape(d1, d0).T."""
    return flat.reshape(-1, d1, d0).transpose(0, 2, 1).reshape(
        flat.shape[:-1] + (d0, d1)) if flat.ndim > 1 else flat.reshape(d1, d0).T


ROW = SIGNS[cfg["n_embd"]]
ROPE_KIND = os.environ.get("ROPE", "neox")
rope = make_rope(ROPE_KIND)
print(f"rope: kind={ROPE_KIND}, n_rot={cfg['n_rot']}, "
      f"freq_base={cfg['freq_base']:g}")


def gdn_layer(x, W, S, SW, conv_hist, _dbg=None):
    """One token through one gated-delta-net layer (models/qwen35.cpp:335)."""
    st, nv, nk, rep = (cfg["state"], cfg["n_v"], cfg["n_k"], cfg["rep"])
    xf = fold(x, ROW)
    qkv = xf @ W["attn_qkv.weight"]                     # [2*key+val]
    z = xf @ W["attn_gate.weight"]                      # [val_dim]
    beta = jax.nn.sigmoid(x @ SW["ssm_beta.weight"])    # [n_v]
    alpha = x @ SW["ssm_alpha.weight"]                  # [n_v]
    # ssm_a already stores -exp(A_log) (qwen35.cpp:484 comment: the product is
    # literally "-A_log.exp() * softplus"), so do NOT exponentiate it again.
    gate = SW["ssm_a"] * jnp.logaddexp(alpha + SW["ssm_dt.bias"], 0.0)

    win = jnp.concatenate([conv_hist, qkv[None]], 0)    # [conv, conv_dim]
    y = jnp.sum(win * SW["ssm_conv1d.weight"], 0)       # causal conv
    y = y * jax.nn.sigmoid(y)                           # SiLU
    hist = win[1:]

    # qk is the first 2*key_dim conv channels: q heads 0..15 then k heads 0..15,
    # each a CONTIGUOUS 128-dim block. In ggml the [head_k_dim, 2*num_k_heads]
    # view has head_k_dim contiguous, so a C-order (heads, dims) reshape must
    # transpose to [st, 2*nk] -- a direct reshape(128, 32) strides instead.
    qk = l2n(y[:2 * cfg["key_dim"]].reshape(2 * nk, st).T, cfg["eps"])
    q, k = qk[:, :nk], qk[:, nk:]                       # [st, n_k]
    v = y[2 * cfg["key_dim"]:].reshape(nv, st).T        # [st, n_v]
    # v-head h uses q/k-head h % n_k (CYCLIC, not block): the fused op indexes
    # iq1 = iv1 % neq1, ik1 = iv1 % nek1 (ops.cpp:11030). Block repeat (h//rep)
    # silently routes every head to the wrong key.
    idx = jnp.arange(nv) % nk
    q, k = q[:, idx], k[:, idx]                         # -> [st, n_v]

    Sg = S * jnp.exp(gate)[None, None, :]
    sk = jnp.einsum("ijv,iv->jv", Sg, k)
    d = (v - sk) * beta[None, :]
    Sg = Sg + jnp.einsum("iv,jv->ijv", k, d)
    # o[j, v]: j = head-local dim (head_v_dim=128), v = v-head  -> [st, nv]
    # in ggml (Fortran) layout, verified exact against attn_output-0.
    o = jnp.einsum("ijv,iv->jv", Sg, q) * (1.0 / jnp.sqrt(jnp.float32(st)))
    if _dbg is not None:
        _dbg["o_attn"] = o

    # gated norm (qwen35.cpp:329-332): final = silu(z) * rms_norm(o, ssm_norm).
    # z is flat [6144]; its ggml [st, nv] view is the Fortran reshape
    # (verified: z_F diff 4.2e-3, and final diff 6.2e-05 with this formula).
    zz = z.reshape(nv, st).T
    _den = jnp.mean(o * o, axis=0, keepdims=True) + cfg["eps"]
    o = o * jax.lax.rsqrt(_den)
    o = o * SW["ssm_norm.weight"][:, None]
    o = jax.nn.sigmoid(zz) * zz * o
    if _dbg is not None and _dbg.get("verbose"):
        _dbg["o_attn_post"] = o
        _dbg["den"] = _den

    # ssm_out permute [hd, h] -> [hd, rep, nk] (model.cpp:2135), then ggml
    # flatten. VERIFIED EXACT against linear_attn_out-0 (diff 3.1e-04):
    # Fortran grouping [i,hk,r] = o[i, hk+r*16], transpose (1,2,0) to
    # [nk,rep,st], C-order flat, fold with signs before R.
    _hk = jnp.arange(nk)[:, None]
    _r = jnp.arange(rep)[None, :]
    Q = o[:, _hk + _r * nk]                # [st, nk, rep], Q[i,hk,r]
    Q = jnp.transpose(Q, (1, 2, 0))        # [nk, rep, st]
    perm_flat = Q.reshape(-1)
    if _dbg is not None:
        _dbg["final"] = o
        _dbg["perm"] = perm_flat
        _dbg["z"] = z
        _dbg["ssm_norm"] = SW["ssm_norm.weight"]
    o_proj = fold(perm_flat, SIGNS[st * nv]) @ W["ssm_out.weight"]
    return o_proj, Sg, hist


def attn_layer(x, W, SW, kc, vc, pos, rope):
    """One token through one full-attention layer (models/qwen35.cpp:335)."""
    H, Hkv, D = cfg["n_head"], cfg["n_head_kv"], cfg["head_dim"]
    xf = fold(x, ROW)
    qg = xf @ W["attn_q.weight"].reshape(-1, 2 * D * H)   # [2*D*H]
    qg = qg.reshape(H, 2, D)                              # q/gate interleaved
    q, g = qg[:, 0, :], qg[:, 1, :]
    q = rms(q, SW["attn_q_norm.weight"], cfg["eps"])
    k = (xf @ W["attn_k.weight"]).reshape(Hkv, D)
    v = (xf @ W["attn_v.weight"]).reshape(Hkv, D)
    k = rms(k, SW["attn_k_norm.weight"], cfg["eps"])
    q, k = rope(q, pos), rope(k, pos)

    # no causal mask needed: the KV cache holds only keys at positions <= pos
    K = jnp.concatenate([kc, k[None]], 0)                 # [T+1, Hkv, D]
    V = jnp.concatenate([vc, v[None]], 0)
    kidx = jnp.arange(H) // (H // Hkv)
    scores = jnp.einsum("hd,thd->ht", q, K[:, kidx, :])
    scores = scores * (1.0 / jnp.sqrt(jnp.float32(D)))
    p = jax.nn.softmax(scores, -1)
    o = jnp.einsum("ht,thd->hd", p, V[:, kidx, :])
    o = o * jax.nn.sigmoid(g)
    o_proj = fold(o.reshape(-1), SIGNS[H * D]) @ W["attn_output.weight"]
    return o_proj, K, V


def ffn_layer(x, W):
    xf = fold(x, ROW)
    g = xf @ W["ffn_gate.weight"]
    u = xf @ W["ffn_up.weight"]
    return fold(jax.nn.silu(g) * u, SIGNS[cfg["n_ff"]]) @ W["ffn_down.weight"]


def forward_one(x, st, pos, rope):
    """One token through the whole stack. Returns (logits, new state)."""
    for il in range(cfg["n_layer"]):
        W, SW = decode_layer(il), SMALLW[il]
        h = rms(x, SW["attn_norm.weight"], cfg["eps"])
        if cfg["is_recr"][il]:
            a, st["s"][il], st["conv"][il] = gdn_layer(
                h, W, st["s"][il], SW, st["conv"][il])
        else:
            a, st["k"][il], st["v"][il] = attn_layer(
                h, W, SW, st["k"][il], st["v"][il], pos, rope)
        x = x + a
        x = x + ffn_layer(rms(x, SW["post_attention_norm.weight"], cfg["eps"]), W)
        del W
    h = rms(x, O_NORM, cfg["eps"])
    return chunked_logits(h), st


def chunked_logits(h, chunk=2048):
    """logits = W @ fold(h), decoded 2048 output rows at a time."""
    xf = fold(h, ROW)
    nb = cfg["n_embd"] // QK
    per = nb * BLK                      # bytes per output token
    parts = []
    for t0 in range(0, cfg["vocab"], chunk):
        t1 = min(t0 + chunk, cfg["vocab"])
        W = decode_ptq1_dev(OUT_RAW[(t0 * per):(t1 * per)]).reshape(t1 - t0, -1)
        parts.append(W @ xf)
        del W
    return jnp.concatenate(parts)


def embed(t):
    """Latent row -> primal embedding (llama-graph.cpp:2453)."""
    row = decode_ptq1_dev(EMB_RAW[(t * EMB_BLK * BLK):((t + 1) * EMB_BLK * BLK)])
    return unfold(row.reshape(-1), ROW)


# output_norm.weight is model-level, not per-layer
raw, dims, dt = ws.read("output_norm.weight")
_n = int(np.prod(dims))
if dt == 143:
    _arr = decode_ptq1_host(np.frombuffer(raw, np.uint8)).reshape(-1)
elif dt == 30:
    _arr = ((np.frombuffer(raw, np.uint16).astype(np.uint32) << 16)
            .view(np.float32).reshape(-1))
else:
    _arr = np.frombuffer(raw, np.float32).reshape(-1)[:_n]
O_NORM = jnp.asarray(np.asarray(_arr[:_n], np.float32).reshape(dims, order="F"))
print(f"output_norm.weight {dims} loaded")


In [ ]:
st = {
    "conv": [jnp.zeros([cfg["conv"] - 1, cfg["conv_dim"]])
             if cfg["is_recr"][i] else None for i in range(cfg["n_layer"])],
    "s": [jnp.zeros([cfg["state"], cfg["state"], cfg["n_v"]])
          if cfg["is_recr"][i] else None for i in range(cfg["n_layer"])],
    "k": [jnp.zeros([0, cfg["n_head_kv"], cfg["head_dim"]])
          if not cfg["is_recr"][i] else None for i in range(cfg["n_layer"])],
    "v": [jnp.zeros([0, cfg["n_head_kv"], cfg["head_dim"]])
          if not cfg["is_recr"][i] else None for i in range(cfg["n_layer"])],
}

ids = tok.encode(PROMPT)
print(f"prompt {PROMPT!r} -> {len(ids)} tokens")

t0 = time.perf_counter()
first = None
for pos, tid in enumerate(ids):
    x = embed(int(tid))
    logits, st = forward_one(x, st, pos, rope)
    first = int(np.argmax(np.asarray(logits)))
    print(f"  prefill {pos}: tok {tid} ({tok.decode([int(tid)])!r}) -> "
          f"next {first} ({tok.decode([first])!r})")
    print(f"    logits: top5 {np.argsort(np.asarray(logits))[-5:][::-1]} "
          f"max={float(np.asarray(logits).max()):.3f} "
          f"min={float(np.asarray(logits).min()):.3f} "
          f"argmax={float(np.asarray(logits)[first]):.3f}")
t_prefill = time.perf_counter() - t0

gen = []
t1 = time.perf_counter()
cur = first
for step in range(N_GEN):
    if cur == tok.eos:
        print(f"  stop: eos at step {step}")
        break
    gen.append(cur)
    x = embed(cur)
    logits, st = forward_one(x, st, len(ids) + step, rope)
    cur = int(np.argmax(np.asarray(logits)))
    print(f"  step {step}: -> {cur} ({tok.decode([cur])!r})")
t_gen = time.perf_counter() - t1

n = len(gen)
print(f"\nprefill {len(ids)} tokens in {t_prefill:.1f}s")
print(f"generated {n} tokens in {t_gen:.1f}s "
      f"({n/t_gen if n else 0:.2f} tok/s)")
print(f"\n=== ANSWER ===\n{tok.decode(gen)}")
